# Chapter 2 — Integrating an orbit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/galactic-dynamics-tutorial/blob/main/notebooks/02_integrating_orbits.ipynb)

**Goal.** Write an orbit integrator from scratch, and learn how to tell whether its output
can be trusted.

**You will learn**
1. why the simplest method (Euler) fails for orbits;
2. the **leapfrog** integrator, the workhorse of galactic dynamics;
3. the difference between *accuracy* (how close the orbit is to the true one) and
   *conservation* (whether energy drifts over many orbits);
4. how to measure an integrator's order of convergence against an exact solution.

**Units.** $G = 1$, as in Chapter 1. We use the Kepler problem ($M = 1$) because its
orbits are known exactly, and the Plummer sphere ($M = b = 1$) from Chapter 1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import brentq

G, M, b = 1.0, 1.0, 1.0

## 2.1 Equations of motion

A star's state is its position $\mathbf x$ and velocity $\mathbf v$. Newton's second law
gives two first-order equations:

$$ \frac{d\mathbf x}{dt} = \mathbf v, \qquad \frac{d\mathbf v}{dt} = \mathbf g(\mathbf x) = -\nabla\Phi . $$

Orbits in a spherical potential stay in a plane (Chapter 1), so two dimensions suffice.
The accelerations of a point mass and a Plummer sphere are

In [ ]:
def accel_kepler(x):
    r = np.linalg.norm(x)
    return -G * M * x / r**3

def accel_plummer(x):
    return -G * M * x / (np.sum(x**2) + b**2)**1.5

def energy(x, v, phi):
    """Energy per unit mass; works on single states or arrays of shape (n, 2)."""
    return 0.5 * np.sum(v**2, axis=-1) + phi(np.linalg.norm(x, axis=-1))

phi_kepler = lambda r: -G * M / r
phi_plummer = lambda r: -G * M / np.sqrt(r**2 + b**2)

As a test orbit we take a Kepler ellipse with semi-major axis $a = 1$ and eccentricity
$e = 0.5$, started at pericentre. Its period is $P = 2\pi\sqrt{a^3/GM} = 2\pi$.

In [ ]:
a, e = 1.0, 0.5
P = 2 * np.pi * np.sqrt(a**3 / (G * M))
x0 = np.array([a * (1 - e), 0.0])
v0 = np.array([0.0, np.sqrt(G * M / a * (1 + e) / (1 - e))])   # speed at pericentre (vis-viva)
E0 = energy(x0, v0, phi_kepler)
print(f'period P = {P:.6f},  energy E = {E0:.6f}  (expected -GM/2a = {-G * M / (2 * a)})')

## 2.2 Euler's method — and why it fails

The most obvious scheme advances both variables using their current rates of change:

$$ \mathbf x_{n+1} = \mathbf x_n + h\,\mathbf v_n, \qquad \mathbf v_{n+1} = \mathbf v_n + h\,\mathbf g(\mathbf x_n). $$

In [ ]:
def euler(accel, x0, v0, h, nsteps):
    x, v = np.empty((nsteps + 1, 2)), np.empty((nsteps + 1, 2))
    x[0], v[0] = x0, v0
    for n in range(nsteps):
        x[n + 1] = x[n] + h * v[n]
        v[n + 1] = v[n] + h * accel(x[n])
    return x, v

h = P / 1000
x_eu, v_eu = euler(accel_kepler, x0, v0, h, 10 * 1000)       # 10 orbits

plt.figure(figsize=(5, 5))
plt.plot(x_eu[:, 0], x_eu[:, 1], lw=0.7)
plt.plot(0, 0, 'k*', ms=12)
plt.gca().set_aspect('equal'); plt.xlabel('$x$'); plt.ylabel('$y$')
plt.title('Euler, 1000 steps per orbit, 10 orbits'); plt.show()
print(f'energy after 10 orbits: {energy(x_eu[-1], v_eu[-1], phi_kepler):.4f}  (started at {E0:.4f})')

The orbit should be a closed ellipse; instead it spirals outwards. Euler's method adds a little
energy at every step, and the errors accumulate. Using smaller steps only delays the problem.

## 2.3 The leapfrog integrator

Leapfrog ("kick–drift–kick") updates position and velocity in a staggered way:

$$
\begin{aligned}
\mathbf v_{n+1/2} &= \mathbf v_n + \tfrac h2\, \mathbf g(\mathbf x_n) && \text{(half kick)}\\
\mathbf x_{n+1}   &= \mathbf x_n + h\, \mathbf v_{n+1/2} && \text{(drift)}\\
\mathbf v_{n+1}   &= \mathbf v_{n+1/2} + \tfrac h2\, \mathbf g(\mathbf x_{n+1}) && \text{(half kick)}
\end{aligned}
$$

It costs one force evaluation per step, the same as Euler (the force at the end of one step is
reused at the start of the next). It has two properties that Euler lacks:
- it is **time-reversible**: running it backwards with $-h$ retraces the path exactly;
- it is **symplectic**: it exactly conserves a slightly perturbed energy, so the energy error
  oscillates but does not grow.

In [ ]:
def leapfrog(accel, x0, v0, h, nsteps):
    """Kick-drift-kick leapfrog. Returns positions and velocities, shape (nsteps+1, dim)."""
    x, v = np.empty((nsteps + 1, len(x0))), np.empty((nsteps + 1, len(x0)))
    x[0], v[0] = x0, v0
    g = accel(x0)
    for n in range(nsteps):
        v_half = v[n] + 0.5 * h * g
        x[n + 1] = x[n] + h * v_half
        g = accel(x[n + 1])
        v[n + 1] = v_half + 0.5 * h * g
    return x, v

x_lf, v_lf = leapfrog(accel_kepler, x0, v0, h, 10 * 1000)
t = np.arange(len(x_lf)) * h

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(x_lf[:, 0], x_lf[:, 1], lw=0.7); ax[0].plot(0, 0, 'k*', ms=12)
ax[0].set_aspect('equal'); ax[0].set_xlabel('$x$'); ax[0].set_ylabel('$y$')
ax[0].set_title('leapfrog, 1000 steps per orbit, 10 orbits')
ax[1].semilogy(t / P, np.abs(energy(x_eu, v_eu, phi_kepler) / E0 - 1), label='Euler')
ax[1].semilogy(t / P, np.abs(energy(x_lf, v_lf, phi_kepler) / E0 - 1), label='leapfrog')
ax[1].set_xlabel('time / orbital period'); ax[1].set_ylabel(r'$|\Delta E / E|$'); ax[1].legend()
plt.tight_layout(); plt.show()

The leapfrog orbit closes on itself; its energy error is largest near pericentre (where the
star moves fastest) and returns to small values every orbit.

**Time reversibility.** Integrate forward, flip the velocity, integrate the same number of
steps again: we should land back at the starting point, up to round-off.

In [ ]:
x_f, v_f = leapfrog(accel_kepler, x0, v0, h, 5000)
x_b, v_b = leapfrog(accel_kepler, x_f[-1], -v_f[-1], h, 5000)
err_rev = np.linalg.norm(x_b[-1] - x0)
print(f'distance from start after going forward and back 5000 steps: {err_rev:.1e}')
assert err_rev < 1e-10

## 2.4 How accurate is it? Convergence against the exact solution

The Kepler orbit is known exactly at any time. With mean motion $n = \sqrt{GM/a^3}$, solve
**Kepler's equation** $\eta - e\sin\eta = n t$ for the eccentric anomaly $\eta$; then

$$ x = a(\cos\eta - e), \qquad y = a\sqrt{1-e^2}\,\sin\eta . $$

In [ ]:
def kepler_position(t, a=a, e=e):
    mean_anomaly = np.sqrt(G * M / a**3) * t
    eta = brentq(lambda s: s - e * np.sin(s) - mean_anomaly, mean_anomaly - 1, mean_anomaly + 1, xtol=1e-15)
    return np.array([a * (np.cos(eta) - e), a * np.sqrt(1 - e**2) * np.sin(eta)])

# check: the leapfrog orbit above, sampled every orbit, against the exact one
x_exact = np.array([kepler_position(ti) for ti in t[::100]])
dx = np.linalg.norm(x_lf[::100] - x_exact, axis=1)
print(f'position error with h = P/1000: after 1 orbit {dx[10]:.1e}, after 10 orbits {dx[-1]:.1e}')

The energy error is bounded, but the position error grows: the integrated star runs slightly
ahead of or behind the true one, and this *phase error* accumulates roughly linearly with time.
Conserving energy is not the same as being accurate.

Now measure the position error at a fixed time $t_{\rm end} = 2.3P$ (not a whole number of
orbits, so that errors cannot cancel by symmetry) while halving the step size. For a method of
order $p$ the error scales as $h^p$: halving $h$ should divide the error by $2^p$.

In [ ]:
t_end = 2.3 * P
x_true = kepler_position(t_end)
steps_per_orbit = np.array([250, 500, 1000, 2000, 4000, 8000])
err_lf, err_eu = [], []
for N in steps_per_orbit:
    nsteps = int(round(t_end / P * N))
    hN = t_end / nsteps
    err_lf.append(np.linalg.norm(leapfrog(accel_kepler, x0, v0, hN, nsteps)[0][-1] - x_true))
    err_eu.append(np.linalg.norm(euler(accel_kepler, x0, v0, hN, nsteps)[0][-1] - x_true))
err_lf, err_eu = np.array(err_lf), np.array(err_eu)
hs = P / steps_per_orbit

plt.figure(figsize=(6, 4.5))
plt.loglog(hs, err_eu, 'o-', label='Euler')
plt.loglog(hs, err_lf, 's-', label='leapfrog')
plt.loglog(hs, err_eu[-1] * (hs / hs[-1]), 'k:', label=r'$\propto h$')
plt.loglog(hs, err_lf[-1] * (hs / hs[-1])**2, 'k--', label=r'$\propto h^2$')
plt.xlabel('step size $h$'); plt.ylabel('position error at $t = 2.3P$'); plt.legend(); plt.show()

ratios = err_lf[:-1] / err_lf[1:]
print('leapfrog error ratios for successive halvings of h:', np.round(ratios, 3))
print('Euler    error ratios                             :', np.round(err_eu[:-1] / err_eu[1:], 3))
assert np.all((ratios > 3.5) & (ratios < 4.5))

Halving $h$ divides the leapfrog error by 4: it is a **second-order** method. Euler is
first order, but its ratios approach 2 only for the smallest steps: at large $h$ the Euler
orbit is so wrong (Section 2.2) that its error no longer follows a power law. A measured order
is meaningful only in this *convergence regime*, where successive ratios agree.

**Choosing a step size.** For accuracy $\epsilon$ in position, the step must satisfy
$h \propto \epsilon^{1/2}$. Check convergence by halving $h$ and seeing whether your answer changes.

## 2.5 Conservation over many orbits: leapfrog vs a "better" integrator

Higher-order methods are more accurate per step. SciPy's `solve_ivp` uses an adaptive
fifth-order Runge–Kutta method (RK45). Compare its energy error with leapfrog over 1000 orbits,
for the same number of force evaluations.

In [ ]:
def rhs(t, w):
    return np.concatenate([w[2:], accel_kepler(w[:2])])

n_orbits = 1000
sol = solve_ivp(rhs, (0, n_orbits * P), np.concatenate([x0, v0]), method='RK45',
                rtol=1e-6, atol=1e-9, dense_output=True)
t_rk = np.linspace(0, n_orbits * P, 50001)
w_rk = sol.sol(t_rk)
dE_rk = np.abs(energy(w_rk[:2].T, w_rk[2:].T, phi_kepler) / E0 - 1)

steps = sol.nfev // n_orbits          # same number of force evaluations as RK45
x_lf, v_lf = leapfrog(accel_kepler, x0, v0, P / steps, n_orbits * steps)
dE_lf = np.abs(energy(x_lf, v_lf, phi_kepler) / E0 - 1)
t_lf = np.arange(len(x_lf)) * P / steps
print(f'RK45 used {sol.nfev} force evaluations; leapfrog uses {steps} steps per orbit')

# the leapfrog error oscillates within each orbit; plot its maximum per orbit
dE_lf_max = dE_lf[1:].reshape(n_orbits, steps).max(axis=1)
plt.figure(figsize=(7, 4))
plt.semilogy(np.arange(1, n_orbits + 1), dE_lf_max, label='leapfrog (maximum in each orbit)')
plt.semilogy(t_rk / P, dE_rk, label='RK45 (rtol = 1e-6)')
plt.xlabel('time / orbital period'); plt.ylabel(r'$|\Delta E / E|$'); plt.legend(); plt.show()

first, last = dE_lf[t_lf < 10 * P].max(), dE_lf[t_lf > (n_orbits - 10) * P].max()
print(f'leapfrog max |dE/E|: first 10 orbits {first:.2e}, last 10 orbits {last:.2e}')
print(f'RK45     |dE/E|    : after 10 orbits {dE_rk[t_rk <= 10 * P][-1]:.2e}, at the end {dE_rk[-1]:.2e}')
print(f'RK45 error first exceeds the leapfrog maximum after {t_rk[np.argmax(dE_rk > last)] / P:.0f} orbits')
assert last < 1.5 * first                               # leapfrog: bounded, no secular drift
assert dE_rk[-1] > 10 * dE_rk[t_rk <= 10 * P][-1]       # RK45: error keeps growing

For the first few hundred orbits RK45 is *more* accurate — it is a higher-order method. But
its energy error grows steadily, while leapfrog's stays bounded no matter how long we
integrate, so eventually RK45 loses. Stars near the centre of a galaxy complete thousands of
orbits in a Hubble time, and an N-body simulation follows millions of stars: a bounded error
is worth more than a small error per step. This is why most N-body codes use leapfrog.

## 2.6 An orbit in the Plummer sphere

Back to the orbit from Chapter 1 §1.3: a star at $r = 1$ moving tangentially at 70% of the
circular speed. There we *predicted* its pericentre ($0.608888$) and apocentre ($1$) from
the effective potential. Now we integrate it and compare.

In [ ]:
r0 = 1.0
vc0 = np.sqrt(G * M * r0**2 / (r0**2 + b**2)**1.5)
x0p, v0p = np.array([r0, 0.0]), np.array([0.0, 0.7 * vc0])
x_p, v_p = leapfrog(accel_plummer, x0p, v0p, 1e-3, 60000)
r_p = np.linalg.norm(x_p, axis=1)
E_p = energy(x_p, v_p, phi_plummer)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].plot(x_p[:, 0], x_p[:, 1], lw=0.6); ax[0].plot(0, 0, 'k*', ms=12)
for rad, ls in [(0.608888, ':'), (1.0, '--')]:
    ax[0].add_patch(plt.Circle((0, 0), rad, fill=False, ls=ls, color='gray'))
ax[0].set_aspect('equal'); ax[0].set_xlabel('$x$'); ax[0].set_ylabel('$y$')
ax[0].set_title('rosette orbit; circles: predicted $r_p$, $r_a$')
ax[1].plot(np.arange(len(r_p)) * 1e-3, r_p)
ax[1].axhline(0.608888, color='gray', ls=':'); ax[1].axhline(1.0, color='gray', ls='--')
ax[1].set_xlabel('time'); ax[1].set_ylabel('$r$')
plt.tight_layout(); plt.show()

print(f'integrated: r_min = {r_p.min():.6f}, r_max = {r_p.max():.6f};  max |dE/E| = {np.max(np.abs(E_p / E_p[0] - 1)):.1e}')
assert abs(r_p.min() - 0.608888) < 1e-5 and abs(r_p.max() - 1.0) < 1e-5

The orbit fills an annulus between the predicted turning points. Unlike the Kepler ellipse it
does not close: each radial oscillation the star advances in angle by the **apsidal angle**
$\Delta\phi$. We measure it from successive apocentres.

In [ ]:
phi = np.unwrap(np.arctan2(x_p[:, 1], x_p[:, 0]))
i_apo = np.where((r_p[1:-1] > r_p[:-2]) & (r_p[1:-1] >= r_p[2:]))[0] + 1
dphi = np.diff(phi[i_apo])
print(f'apsidal angle = {dphi.mean():.4f} rad = {np.degrees(dphi.mean()):.1f} deg  (scatter {dphi.std():.1e})')
assert np.pi < dphi.mean() < 2 * np.pi

The apsidal angle lies between $\pi$ (a harmonic potential, where orbits are centred ellipses)
and $2\pi$ (Kepler, where orbits close after one radial period). Chapter 3 explores how it
depends on the orbit, and introduces AGAMA to do these integrations for us.

## Summary

- Euler's method adds energy every step; orbits spiral out.
- Leapfrog (kick–drift–kick) is second order, time-reversible and symplectic: its energy error
  stays bounded over arbitrarily many orbits.
- Check an integrator against an exact solution and measure its order: halve $h$, the error
  should drop by $2^p$.
- Bounded energy error is not the same as accuracy: the phase error still grows.
- A higher-order method is not automatically better for long integrations: RK45's energy drifts.
- Integrated turning points agree with the effective-potential prediction of Chapter 1.

## Exercises

1. Write a classical fourth-order Runge–Kutta (RK4) integrator with a fixed step. Measure its
   order with the method of §2.4, then plot its energy error over 1000 orbits as in §2.5.
2. Implement the "drift–kick–drift" variant of leapfrog. Is it still second order?
3. Make the leapfrog step adaptive, $h \propto r^{3/2}$ (smaller near pericentre). What happens
   to the long-term energy error, and why? (Hint: is the scheme still time-reversible?)
4. Integrate the Kepler orbit with $e = 0.9$. How many steps per orbit do you need for a
   position error of $10^{-4}$ after one orbit?
5. Compute the apsidal angle of Plummer orbits with the same apocentre $r = 1$ and
   tangential speeds from 10% to 99% of $v_c$. Plot $\Delta\phi$ against $v/v_c$.

## References

- J. Binney & S. Tremaine, *Galactic Dynamics*, 2nd ed. (Princeton, 2008), §3.4.
- J. Bovy, *Dynamics and Astrophysics of Galaxies*, chapter 4: https://galaxiesbook.org
- E. Hairer, C. Lubich & G. Wanner (2003), "Geometric numerical integration illustrated by
  the Störmer–Verlet method", Acta Numerica 12, 399: https://doi.org/10.1017/S0962492902000144